In [1]:
# Run to install dependencies

! pip install sqlalchemy
! pip install pymysql
! pip install pandas


# **Make sure to run "Data download" notebook first so that the csv is picked up from the correct location**

In [2]:
from sqlalchemy import create_engine, text  # Import necessary modules
import pandas as pd # For dataframes (optional but useful)

# Construct the database
engine = create_engine("sqlite:///IMDB.db")  # Creates a file named your_database.db

# Test the connection
try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Error connecting to database: {e}")
    exit() # Exit if connection fails


Connection successful!


In [3]:
# Load the CSV file into a Pandas DataFrame
csv_file = 'data/IMDB TMDB Movie Metadata Big Dataset (1M).csv'
df = pd.read_csv(csv_file)

print("DataFrame loaded successfully:")
print(df.head()) # Display the first few rows to verify

DataFrame loaded successfully:
       id            title  vote_average  vote_count    status release_date  \
0   27205        Inception         8.364       34495  Released   2010-07-15   
1  157336     Interstellar         8.417       32571  Released   2014-11-05   
2     155  The Dark Knight         8.512       30619  Released   2008-07-16   
3   19995           Avatar         7.573       29815  Released   2009-12-15   
4   24428     The Avengers         7.710       29166  Released   2012-04-25   

      revenue  runtime  adult                     backdrop_path  ...  \
0   825532764      148  False  /8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg  ...   
1   701729206      169  False  /pbrkL804c8yAv3zBZR4QPEafpAR.jpg  ...   
2  1004558444      152  False  /nMKdUUepR0i5zn0y1T4CsSB5chy.jpg  ...   
3  2923706026      162  False  /vL5LR6WdxWPjLPFRLe133jXWsh5.jpg  ...   
4  1518815515      143  False  /9BBTo63ANSmhC4e6r62OJFuK2GL.jpg  ...   

                Star3               Star4             Writer 

In [4]:
# Write the DataFrame to the SQLite database
table_name = 'IMDB_Metadata'

try:
    df.to_sql(table_name, engine, if_exists='replace', index=False) #if_exists handles existing tables
    print(f"Data from {csv_file} successfully written to table '{table_name}'.")

except Exception as e:
    print(f"Error writing data to database: {e}")


Data from data/IMDB TMDB Movie Metadata Big Dataset (1M).csv successfully written to table 'IMDB_Metadata'.


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT title, vote_average FROM IMDB_Metadata WHERE vote_average > 8.0"))  # Replace with your table name
    for row in result:
        print(row)

('Inception', 8.364)
('Interstellar', 8.417)
('The Dark Knight', 8.512)
('Avengers: Infinity War', 8.255)
('Fight Club', 8.438)
('Pulp Fiction', 8.488)
('Forrest Gump', 8.477)
('Django Unchained', 8.171)
('The Shawshank Redemption', 8.702)
('Avengers: Endgame', 8.263)
('The Matrix', 8.206)
('Joker', 8.168)
('The Lord of the Rings: The Fellowship of the Ring', 8.402)
('The Lord of the Rings: The Return of the King', 8.474)
('Shutter Island', 8.2)
('The Wolf of Wall Street', 8.035)
('Inglourious Basterds', 8.215)
('The Lord of the Rings: The Two Towers', 8.385)
('Harry Potter and the Prisoner of Azkaban', 8.02)
('Se7en', 8.368)
('Harry Potter and the Deathly Hallows: Part 2', 8.105)
('Star Wars', 8.204)
('The Godfather', 8.707)
('Back to the Future', 8.314)
('Coco', 8.222)
('WALL·E', 8.078)
('The Lion King', 8.256)
('Gladiator', 8.209)
('The Truman Show', 8.133)
('Parasite', 8.515)
('The Shining', 8.218)
('The Intouchables', 8.277)
('The Imitation Game', 8.007)
('The Green Mile', 8.507)


In [6]:
import pandas as pd
import numpy as np

print("Loading data...")
df = pd.read_csv('data/IMDB TMDB Movie Metadata Big Dataset (1M).csv')

print(f"\n{'='*70}")
print("DATASET OVERVIEW")
print(f"{'='*70}")
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

print(f"\n{'='*70}")
print("KEY STATISTICS")
print(f"{'='*70}")

# Try to get budget and revenue - try different column names
budget_col = None
revenue_col = None

# Find columns
for col in df.columns:
    if 'budget' in col.lower() and budget_col is None:
        budget_col = col
    if 'revenue' in col.lower() and revenue_col is None:
        revenue_col = col

print(f"\nBudget column: {budget_col}")
print(f"Revenue column: {revenue_col}")

# If we found them, calculate ROI
if budget_col and revenue_col:
    print(f"\n{'='*70}")
    print("DATA CLEANING")
    print(f"{'='*70}")
    
    # Count nulls
    budget_nulls = df[budget_col].isna().sum()
    revenue_nulls = df[revenue_col].isna().sum()
    
    print(f"\nNull values:")
    print(f"  {budget_col}: {budget_nulls:,}")
    print(f"  {revenue_col}: {revenue_nulls:,}")
    
    # Clean data
    df_clean = df[(df[budget_col] > 0) & (df[revenue_col] > 0)].copy()
    
    removed = len(df) - len(df_clean)
    removal_pct = (removed / len(df)) * 100
    
    print(f"\nOriginal records: {len(df):,}")
    print(f"After cleaning (budget > 0 AND revenue > 0): {len(df_clean):,}")
    print(f"Removed: {removed:,} ({removal_pct:.1f}%)")
    
    print(f"\n{'='*70}")
    print("ROI ANALYSIS")
    print(f"{'='*70}")
    
    # Calculate ROI
    df_clean['roi'] = (df_clean[revenue_col] - df_clean[budget_col]) / df_clean[budget_col]
    
    print(f"\nROI Statistics:")
    print(f"  Average ROI: {df_clean['roi'].mean():.4f}")
    print(f"  Median ROI: {df_clean['roi'].median():.4f}")
    print(f"  Min ROI: {df_clean['roi'].min():.4f}")
    print(f"  Max ROI: {df_clean['roi'].max():.4f}")
    print(f"  Std Dev: {df_clean['roi'].std():.4f}")
    
    # Profitable vs unprofitable
    profitable = len(df_clean[df_clean['roi'] > 0])
    unprofitable = len(df_clean[df_clean['roi'] <= 0])
    
    print(f"\nProfitable movies (ROI > 0): {profitable:,} ({profitable/len(df_clean)*100:.1f}%)")
    print(f"Loss-making movies (ROI <= 0): {unprofitable:,} ({unprofitable/len(df_clean)*100:.1f}%)")

print(f"\n{'='*70}")
print("KEY FINDINGS FOR YOUR REPORT")
print(f"{'='*70}")

if budget_col and revenue_col:
    print(f"""
 Dataset: {len(df):,} total movies
 After cleaning: {len(df_clean):,} valid records
 Data removed: {removed:,} records ({removal_pct:.1f}%)
 Average ROI: {df_clean['roi'].mean():.4f} ({df_clean['roi'].mean()*100:.1f}% profit margin)
 ROI Range: {df_clean['roi'].min():.2f}x to {df_clean['roi'].max():.2f}x
 Profitable: {profitable:,} movies ({profitable/len(df_clean)*100:.1f}%)

""")
else:
    print(f"""
Could not find budget or revenue columns.
Available columns: {list(df.columns)}
""")

print("Done!")

Loading data...

DATASET OVERVIEW
Total records: 1,072,255
Total columns: 42
Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'release_year', 'Director', 'AverageRating', 'Poster_Link', 'Certificate', 'IMDB_Rating', 'Meta_score', 'Star1', 'Star2', 'Star3', 'Star4', 'Writer', 'Director_of_Photography', 'Producers', 'Music_Composer', 'genres_list', 'Cast_list', 'overview_sentiment', 'all_combined_keywords']

KEY STATISTICS

Budget column: budget
Revenue column: revenue

DATA CLEANING

Null values:
  budget: 0
  revenue: 0

Original records: 1,072,255
After cleaning (budget > 0 AND revenue > 0): 13,759
Removed: 1,058,496 (98.7%)

ROI ANALYSIS

ROI Statistics:
  Average ROI: 294935.0392
  Median ROI: 0.5462
  M

In [ ]:
import sys
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
from pyspark import SparkContext

sc = SparkContext("local", "IMDB SQLLite Query")


with engine.connect() as conn:
    result = conn.execute(text("SELECT title, vote_average FROM IMDB_Metadata"))  # Replace with your table name

# Parallelize the results in Spark RDD
rows = result.fetchall()
rdd = sc.parallelize(rows)

# Filter like SQL WHERE
high_rated_rdd = rdd.filter(lambda x: x[1] > 8.0)

# Show top 20 results
for row in high_rated_rdd.take(20):
    print(row)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=IMDB SQLLite Query, master=local) created by __init__ at C:\Users\pouri\AppData\Local\Temp\ipykernel_19460\1716986941.py:8 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 64194)
Traceback (most recent call last):
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\accumulators.py", line 303, in handle
    poll(accum_updates)
  File "C:\Users\pouri\AppData\Local\Programs\Python\Python311\Lib

In [2]:
import os
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_DRIVER_HOST"] = "127.0.0.1"
from pyspark import SparkContext

sc = SparkContext("local", "IMDB SQLLite Query")

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=IMDB SQLLite Query, master=local) created by __init__ at C:\Users\pouri\AppData\Local\Temp\ipykernel_17348\4055197418.py:8 